## Automatic annotation using SAM3 text-prompt model

In [ ]:
import json
import sys
import torch
import numpy as np
from pathlib import Path
from PIL import Image
from skimage import measure
from tqdm import tqdm

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "src":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# SAM3 lives under pkg/sam3/
SAM3_DIR = PROJECT_ROOT / "pkg" / "sam3"
if str(SAM3_DIR) not in sys.path:
    sys.path.insert(0, str(SAM3_DIR))

In [ ]:
from sam3.model_builder import build_sam3_image_model
from sam3.model.sam3_image_processor import Sam3Processor
import os
from pathlib import Path

os.chdir(Path.cwd().resolve().parent)
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False

# Build model and processor (resolution=1008 for best quality)
model = build_sam3_image_model()
processor = Sam3Processor(model, confidence_threshold=0.35)
print("✅ SAM3 model loaded")

### Configure class → text prompt mapping

In [ ]:
# class_id → (Chinese label, English text prompt for SAM3)
class_config = {
    0: ("red box", "red box"),
    1: ("small object", "small object"),
    2: ("green small box", "green small box"),
    3: ("toy car", "toy car"),
}

target_folder = PROJECT_ROOT / "datasets/giftbox_task/images"
annotation_save_path = PROJECT_ROOT / "datasets/giftbox_task/annotations/annotations.json"

target_paths = sorted(target_folder.glob("*"))
print(f"Found {len(target_paths)} target images")
print(f"Classes: {[v[0] for v in class_config.values()]}")

## Run SAM3 text-prompt inference & save COCO annotations

In [ ]:
def sam3_predict_mask(processor, image, text_prompt):
    """
    Run SAM3 text-prompt on a single image, return mask contours + boxes.

    Returns list of (segmentation_polygon_flat, bbox_xywh, score)
    """
    state = processor.set_image(image)
    state = processor.set_text_prompt(prompt=text_prompt, state=state)

    results = []
    if "masks" not in state or len(state["scores"]) == 0:
        return results

    boxes = state["boxes"].cpu().numpy()  # [N, 4] XYXY in pixel coords
    masks = state["masks"].cpu().numpy()  # [N, 1, H, W] boolean
    scores = state["scores"].cpu().numpy()

    for i in range(len(scores)):
        mask_np = masks[i].squeeze(0).astype(np.uint8)
        if not mask_np.any():
            continue

        # Find contours from mask
        contours = measure.find_contours(mask_np.astype(np.float64), 0.5)
        if not contours:
            continue
        contour = max(contours, key=len)
        # Convert (row, col) → (x, y)
        contour_xy = np.fliplr(contour)
        segmentation = contour_xy.flatten().tolist()

        x1, y1, x2, y2 = boxes[i]
        bbox_w = x2 - x1
        bbox_h = y2 - y1

        # Compute area via polygon shoelace formula
        poly = contour_xy
        area = 0.5 * abs(np.dot(poly[:, 0], np.roll(poly[:, 1], 1)) -
                         np.dot(poly[:, 1], np.roll(poly[:, 0], 1)))

        results.append((segmentation, [float(x1), float(y1), float(bbox_w), float(bbox_h)], float(area)))

    return results


def make_coco_annotation_sam3(target_paths, class_config, processor, save_path, min_area=0):
    """
    Run SAM3 on all images, one text prompt per class, and save COCO JSON.
    """
    coco_output = {
        "images": [],
        "annotations": [],
        "categories": [
            {"id": cid, "name": label, "supercategory": "object"}
            for cid, (label, _) in class_config.items()
        ],
    }

    ann_id = 1
    total_objs = 0

    # Build images list
    for img_id, target_path in enumerate(target_paths, start=1):
        target_img = Image.open(target_path).convert("RGB")
        w, h = target_img.size
        coco_output["images"].append({
            "id": img_id,
            "file_name": target_path.name,
            "width": w,
            "height": h,
        })

    # Run inference: per image, per class
    for img_id, target_path in enumerate(tqdm(target_paths, desc="SAM3 inference"), start=1):
        image = Image.open(target_path).convert("RGB")

        for class_id, (label, text_prompt) in class_config.items():
            segms = sam3_predict_mask(processor, image, text_prompt)
            for seg, bbox, area in segms:
                if area < min_area:
                    continue
                coco_output["annotations"].append({
                    "id": ann_id,
                    "image_id": img_id,
                    "category_id": class_id,
                    "segmentation": [seg],
                    "bbox": bbox,
                    "area": area,
                    "iscrowd": 0,
                })
                ann_id += 1
                total_objs += 1

    # Save JSON
    save_path = Path(save_path)
    save_path.parent.mkdir(parents=True, exist_ok=True)
    with open(save_path, "w", encoding="utf-8") as f:
        json.dump(coco_output, f, indent=2, ensure_ascii=False)

    # Save labels.txt
    labels_path = save_path.parent / "labels.txt"
    with open(labels_path, "w", encoding="utf-8") as f_labels:
        for cid in sorted(class_config.keys()):
            f_labels.write(f"{class_config[cid][0]}\n")

    print(f"\n✅ COCO annotation saved to {save_path}")
    print(f"   images={len(coco_output['images'])}, "
          f"annotations={len(coco_output['annotations'])}, "
          f"categories={len(coco_output['categories'])}")
    print(f"   total objects found: {total_objs}")

In [ ]:
# Run auto-annotation
if not annotation_save_path.exists():
    annotation_save_path.parent.mkdir(parents=True, exist_ok=True)
    annotation_save_path.touch()

make_coco_annotation_sam3(
    target_paths=target_paths,
    class_config=class_config,
    processor=processor,
    save_path=annotation_save_path,
    min_area=0,
)